# 01_02 - Tech-Viet Translation Data Collection

## Mục tiêu

Notebook này thực hiện bước **Data Collection** cho source:

**lightontech/tech-viet-translation**

Các nhiệm vụ:

1. Kiểm tra môi trường
2. Xác định project root
3. Cấu hình source
4. Download dataset
5. Convert dữ liệu sang DataFrame
6. Inspect raw data
7. Thống kê raw data
8. Xác định technology candidate
9. Tổng hợp audit summary
10. Lưu raw JSONL
11. Lưu raw Parquet
12. Lưu audit summary
13. Lưu metadata
14. Final verification
15. Ghi trạng thái notebook

> Lưu ý:
> - Notebook này không cleaning dữ liệu
> - Không deduplication
> - Không train/validation/test split
> - Không overwrite raw data
> - Source-level technology candidate chưa đồng nghĩa với usable IT corpus
> - `usable_count` chưa được xác định


In [ ]:
import sys
import os
from pathlib import Path

print("Python:", sys.version)
print("Working directory:", os.getcwd())

In [ ]:
import datasets
import pandas as pd
import httpx

print("datasets:", datasets.__version__)
print("pandas:", pd.__version__)
print("httpx:", httpx.__version__)

In [ ]:
from pathlib import Path

CURRENT_DIR = Path.cwd().resolve()


def find_project_root(start_path: Path) -> Path:
    """
    Tìm project root bằng cách kiểm tra:
    - data/
    - notebooks/ hoặc notebook/
    """
    candidates = [start_path] + list(start_path.parents)

    for path in candidates:
        if (
            (path / "data").is_dir()
            and (
                (path / "notebooks").is_dir()
                or (path / "notebook").is_dir()
            )
        ):
            return path

    raise FileNotFoundError(
        "Không tìm thấy project root. "
        "Hãy kiểm tra lại vị trí notebook."
    )


PROJECT_ROOT = find_project_root(CURRENT_DIR)

print("Project root:")
print(PROJECT_ROOT)

In [ ]:
SOURCE_NAME = "tech-viet-translation"

SOURCE_SHORT_NAME = "tech_viet_translation"

SOURCE_URL = (
    "https://huggingface.co/datasets/"
    "lightontech/tech-viet-translation"
)

LANGUAGE_PAIR = "en-vi"

DOMAIN = "IT"

DATASET_VERSION = "main"

DOWNLOAD_METHOD = "Hugging Face datasets.load_dataset"

COLLECTION_DATE = pd.Timestamp.now().strftime("%Y-%m-%d")

RAW_DIR = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / SOURCE_SHORT_NAME
)

RAW_DIR.mkdir(parents=True, exist_ok=True)

print("Source:", SOURCE_NAME)
print("Source URL:", SOURCE_URL)
print("Language pair:", LANGUAGE_PAIR)
print("Domain:", DOMAIN)
print("Dataset version:", DATASET_VERSION)
print("Raw directory:", RAW_DIR)

In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "lightontech/tech-viet-translation"
)

print(dataset)

In [ ]:
df = dataset["train"].to_pandas()

print("Shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

In [ ]:
print("First 5 rows:")
display(df.head())

print("\nRandom 5 rows:")
display(df.sample(5, random_state=42))

In [ ]:
raw_count = len(df)

missing_values = df.isna().sum()

duplicate_count = df.duplicated().sum()

print("Raw count:", raw_count)

print("\nMissing values:")
display(missing_values)

print("\nDuplicate rows:")
print(duplicate_count)

print("\nColumn statistics:")

for column in df.columns:
    print(f"\n{column}")
    print("Non-null:", df[column].notna().sum())
    print("Unique:", df[column].nunique(dropna=False))

In [ ]:
df_tech_candidate = df.copy()

technology_candidate_count = len(
    df_tech_candidate
)

non_technology_count = (
    raw_count - technology_candidate_count
)

print("Raw rows:", raw_count)

print(
    "Technology candidate rows:",
    technology_candidate_count
)

print(
    "Non-technology rows:",
    non_technology_count
)

In [ ]:
audit_summary = {
    "source": SOURCE_NAME,
    "raw_count": int(raw_count),
    "candidate_count": int(technology_candidate_count),
    "non_technology_count": int(non_technology_count),
    "missing_values": {
        str(key): int(value)
        for key, value in missing_values.items()
    },
    "duplicate_count": int(duplicate_count),
    "columns": [
        str(column)
        for column in df.columns
    ],
    "candidate_rule": (
        "All rows from the source are retained as "
        "technology candidates based on the source "
        "identity/category; this does not establish "
        "final usable IT status."
    )
}

audit_summary

In [ ]:
raw_jsonl_path = (
    RAW_DIR
    / f"{SOURCE_SHORT_NAME}_raw.jsonl"
)
raw_parquet_path = RAW_DIR / f"{SOURCE_SHORT_NAME}_raw.parquet"
audit_path = RAW_DIR / "audit_summary.json"
metadata_path = RAW_DIR / "metadata.json"

# Phase 01 tạo snapshot RAW một lần; không ghi đè artifact đã có.
phase_01_output_paths = [raw_jsonl_path, raw_parquet_path, audit_path, metadata_path]
existing_outputs = [path for path in phase_01_output_paths if path.exists()]
missing_outputs = [path for path in phase_01_output_paths if not path.exists()]
if existing_outputs and missing_outputs:
    raise RuntimeError(
        "Phát hiện RAW snapshot chưa đầy đủ; không được ghi đè hay tiếp tục. \n"
        f"Existing: {[str(path) for path in existing_outputs]}\n"
        f"Missing: {[str(path) for path in missing_outputs]}"
    )

write_raw_snapshot = not existing_outputs
if write_raw_snapshot:
    print("No existing RAW snapshot found; creating a new immutable snapshot.")
else:
    print("Complete RAW snapshot already exists; preserving it and skipping writes.")

if write_raw_snapshot:
    df.to_json(
        raw_jsonl_path,
        orient="records",
        lines=True,
        force_ascii=False
    )

print("Saved:" if write_raw_snapshot else "Preserved existing:")
print(raw_jsonl_path)

In [ ]:
raw_parquet_path = (
    RAW_DIR
    / f"{SOURCE_SHORT_NAME}_raw.parquet"
)

if write_raw_snapshot:
    df.to_parquet(
        raw_parquet_path,
        index=False
    )

print("Saved:" if write_raw_snapshot else "Preserved existing:")
print(raw_parquet_path)

In [ ]:
import json

audit_path = RAW_DIR / "audit_summary.json"

if write_raw_snapshot:
    with open(
        audit_path,
        "w",
        encoding="utf-8"
    ) as f:
        json.dump(
            audit_summary,
            f,
            ensure_ascii=False,
            indent=2
        )

print("Saved:" if write_raw_snapshot else "Preserved existing:")
print(audit_path)

In [ ]:
metadata = {
    "source": SOURCE_NAME,
    "source_url": SOURCE_URL,
    "version_revision": DATASET_VERSION,
    "collection_date": COLLECTION_DATE,
    "download_method": DOWNLOAD_METHOD,
    "language_pair": LANGUAGE_PAIR,
    "domain": DOMAIN,
    "subcategory": None,
    "raw_count": int(raw_count),
    "candidate_count": int(technology_candidate_count),
    "usable_count": None,
    "notes": 'Raw dataset collected from the public source. Technology candidate count is source-level; final IT usability is determined in later pipeline phases.',
}

metadata_path = RAW_DIR / "metadata.json"
if write_raw_snapshot:
    with open(metadata_path, "w", encoding="utf-8") as file:
        json.dump(metadata, file, ensure_ascii=False, indent=2)

print("Saved:" if write_raw_snapshot else "Preserved existing:")
print(metadata_path)


In [ ]:
expected_files = [
    raw_jsonl_path,
    raw_parquet_path,
    audit_path,
    metadata_path
]

verification_results = {}

for file_path in expected_files:
    verification_results[file_path.name] = file_path.is_file()

print("Final verification:\n")

for file_name, exists in verification_results.items():
    print(
        f"{file_name:45}"
        f"{'OK' if exists else 'MISSING'}"
    )

verification_passed = all(
    verification_results.values()
)

print("\nVerification passed:", verification_passed)

# Data Collection Status

Source:

**lightontech/tech-viet-translation**

| Metric | Value |
|---|---:|
| Raw rows | 101,767 |
| Technology candidate rows | 101,767 |
| Non-technology rows | 0 |
| Usable IT rows | TBD |

## Completed in this notebook

- [x] Environment checked
- [x] Project root identified
- [x] Source configured
- [x] Dataset downloaded
- [x] Raw schema inspected
- [x] Raw statistics recorded
- [x] Technology candidate identified
- [x] Raw JSONL saved
- [x] Raw Parquet saved
- [x] Audit summary saved
- [x] Metadata saved
- [x] Output files verified

## Not completed in this notebook

- [ ] Full language check
- [ ] Alignment check
- [ ] Data cleaning
- [ ] Deduplication
- [ ] Quality/noise assessment
- [ ] IT subdomain classification
- [ ] Final usable IT count
- [ ] Train / validation / test split

## Interpretation

`101,767` is the raw source count.

`101,767` is also retained as the source-level technology candidate count
because this source is explicitly published as a technology-focused
English-Vietnamese translation dataset.

This does **not** mean that all 101,767 pairs are final usable IT pairs.

`usable_count = TBD`

The final usable IT corpus must be determined only after the subsequent
audit, cleaning and IT-filtering stages.

> Raw data under `data/raw/` must remain unchanged.
>
> This notebook does not produce the final IT corpus.
>
> before the source is used in the final artifact.
